In [1]:
%load_ext autoreload
%autoreload 2

In [104]:
import legogp as lego
import objax
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import batchjax

In [93]:
# generate data
P = 10

x = np.linspace(0, 1, 100)
y = np.sin(x*10)

X = x[:, None]
Y = y[:, None]

print(X.shape, Y.shape)

(100, 1) (100, 1)


In [99]:
YP = np.hstack([Y*p for p in range(P)])

In [100]:
# Construct model

m_list = objax.ModuleList([
    lego.models.GP(X, Y+p)
    for p in range(P)
])

In [101]:
def get_objective(y_p, m, m2):
    return jnp.sum(y_p), m.get_objective(), m2.get_objective()

In [102]:
%%timeit -r 1 -n 1
# Manual 
val_list = []
for p in range(P):
    val_list.append(
        get_objective(YP[:, p], m_list[p], m_list[p])
    )
val = np.vstack(val_list)
print(val)

[[  0.       116.65031  116.65031 ]
 [ 17.919315 118.19217  118.19217 ]
 [ 35.83863  120.898834 120.898834]
 [ 53.757942 124.7703   124.7703  ]
 [ 71.67726  129.80658  129.80658 ]
 [ 89.59657  136.00764  136.00764 ]
 [107.515884 143.37354  143.37354 ]
 [125.43518  151.9042   151.9042  ]
 [143.35452  161.5997   161.5997  ]
 [161.27382  172.45999  172.45999 ]]
212 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [120]:
%%timeit -r 1 -n 1
# vmap approach
val = batchjax.batch_fn(get_objective, [YP, m_list, m_list], axes=[1, 0, 0])
print(val)

[[  0.       116.6503   116.6503  ]
 [ 17.919315 118.19217  118.19217 ]
 [ 35.83863  120.89884  120.89884 ]
 [ 53.757942 124.770294 124.770294]
 [ 71.67726  129.80656  129.80656 ]
 [ 89.59657  136.00766  136.00766 ]
 [107.515884 143.37352  143.37352 ]
 [125.435196 151.9042   151.9042  ]
 [143.35452  161.5997   161.5997  ]
 [161.27383  172.45999  172.45999 ]]
78.1 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [123]:
%%timeit -r 1 -n 1
val = batchjax.batch_or_loop(get_objective, [YP, m_list, m_list], axes=[1, 0, 0], batch_flag=False)
print(val)

[[  0.       116.65031  116.65031 ]
 [ 17.919315 118.19217  118.19217 ]
 [ 35.83863  120.898834 120.898834]
 [ 53.757942 124.7703   124.7703  ]
 [ 71.67726  129.80658  129.80658 ]
 [ 89.59657  136.00764  136.00764 ]
 [107.515884 143.37354  143.37354 ]
 [125.43518  151.9042   151.9042  ]
 [143.35452  161.5997   161.5997  ]
 [161.27382  172.45999  172.45999 ]]
218 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
